# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset defined by a [Croissant schema](https://mlcommons.org/croissant/) using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is provided at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` and required libraries are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading

Load the dataset's metadata and read records using `mlcroissant`. We'll define the dataset schema URL and create a Dataset instance.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print details
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets (tables) and the fields within them by `@id`. We'll list all `recordSet` entries defined for this dataset and show their available field `@id`s.

> **Note:** All entities are referenced by their Croissant `@id`, ensuring reproducibility and precision.

In [ ]:
# List all available record sets and their fields
df_record_sets = []
for record_set in dataset.record_sets:
    rs_dict = {
        '@id': record_set.id,
        'name': record_set.name,
        'fields': [field.id for field in record_set.fields] if hasattr(record_set, 'fields') else []
    }
    df_record_sets.append(rs_dict)

record_sets_overview = pd.DataFrame(df_record_sets)
print('Available record sets:')
display(record_sets_overview)

# Show sample records for each record set by @id
print('\nSample of available records by record set:')
for rec in dataset.record_sets:
    print(f"\nRECORD SET: {rec.id} ({rec.name})")
    try:
        for i, row in enumerate(dataset.records(record_set=rec.id)):
            print(row)
            if i == 1: break  # Show first two
    except Exception as e:
        print(f"Could not iterate records: {e}")

## 3. Data Extraction

We'll load data from the record set(s) of interest into pandas DataFrames for analysis. All record sets in the dataset are referenced by their `@id`.

In [ ]:
# Build list of all record_set @id's
record_sets_ids = [rs['@id'] for rs in df_record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id}: loaded {df.shape[0]} records and {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# For analysis, choose the first available record set with tabular data
for rec_id, df in dataframes.items():
    if df.shape[0] > 0:
        chosen_record_set_id = rec_id
        break

print(f"\nColumns (@id) in chosen record set '{chosen_record_set_id}':")
print(dataframes[chosen_record_set_id].columns.tolist())
display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's analyze a numeric field. We'll:
- Filter records with value above a threshold
- Normalize the numeric column
- (If a suitable field exists) group by a categorical field

> *All fields are referenced by their `@id` as defined in the Croissant schema.*

In [ ]:
# Identify numeric and group fields by inspecting the DataFrame
df = dataframes[chosen_record_set_id]

# Try to automatically detect a numeric column (float or int)
numeric_field_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else None

if numeric_field_id is None:
    print('No numeric field detected for EDA.')
else:
    print(f"Using numeric field '@id': {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    print(f"Applying filter: {numeric_field_id} > {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (total: {filtered_df.shape[0]}):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to automatically detect a group field (categorical, with <20 unique values)
    group_field_candidates = [col for col in df.columns if (df[col].dtype == object and df[col].nunique() < 20 and col != numeric_field_id)]
    group_field_id = group_field_candidates[0] if group_field_candidates else None

    if group_field_id:
        print(f"\nGrouping by field '@id': {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print('No suitable group field found for grouping.')

## 5. Visualization

Visualize the distribution of the numeric field and the effect of grouping by a categorical field if available.

In [ ]:
# Visualize numeric field distribution and group comparison
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.barplot(
            x=group_field_id, y=numeric_field_id,
            data=df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Loaded a Croissant-structured dataset using `mlcroissant`
- Explored available record sets and fields by their `@id`
- Extracted data into pandas DataFrames
- Performed simple EDA, implemented basic normalization and grouping
- Visualized field distributions

Refer to the dataset's documentation and Croissant schema for further analysis ideas. All data manipulations reference the precise `@id` for robust reproducibility.